# RabTech Task 2 — Data Ingestion, Cleaning & Preprocessing with Pandas

**Dataset:** Retail Store Sales (dirty dataset)  
**Objective:** Clean a 10,000+ row retail transaction dataset, document before/after quality, handle missing values and outliers, correct data types, engineer date and margin-related features, and export `clean_dataset.csv`.

> **Important business note:** The supplied dataset does not contain an actual cost/profit column. Therefore, the notebook creates an **illustrative estimated profit margin proxy** using a clearly stated 70% cost-rate assumption. This must not be interpreted as actual accounting profit.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_FILE = "retail_store_sales.csv"
df = pd.read_csv(DATA_FILE)

print("Raw shape:", df.shape)
display(df.head())
display(df.info())


## 1. Before-cleaning profile

In [ ]:
profile_before = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=True)
})
display(profile_before)

print("Exact duplicate rows:", df.duplicated().sum())


## 2. Cleaning and preprocessing

In [ ]:
# Standardize column names
df.columns = [c.strip().replace(" ", "_").replace("-", "_") for c in df.columns]

# Correct data types
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], errors="coerce")
for c in ["Price_Per_Unit", "Quantity", "Total_Spent"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Strip text fields
text_cols = ["Transaction_ID","Customer_ID","Category","Item","Payment_Method","Location"]
for c in text_cols:
    df[c] = df[c].astype("string").str.strip()

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Convert impossible numeric values to missing
df.loc[df["Price_Per_Unit"] <= 0, "Price_Per_Unit"] = np.nan
df.loc[df["Quantity"] <= 0, "Quantity"] = np.nan
df.loc[df["Total_Spent"] < 0, "Total_Spent"] = np.nan

# Categorical imputation
for c in ["Item","Payment_Method","Category","Location"]:
    df[c] = df[c].fillna("Unknown")
df["Discount_Applied"] = df["Discount_Applied"].astype("string").fillna("Unknown")

# Numeric imputation using category median, then overall median
for c in ["Price_Per_Unit", "Quantity"]:
    df[c] = df.groupby("Category")[c].transform(lambda s: s.fillna(s.median()))
    df[c] = df[c].fillna(df[c].median())
df["Quantity"] = df["Quantity"].round().astype(int)

# Reconstruct missing total spent from price × quantity
df["Total_Spent"] = df["Total_Spent"].fillna(df["Price_Per_Unit"] * df["Quantity"])


## 3. Outlier detection and handling

In [ ]:
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_summary = []
for c in ["Price_Per_Unit", "Quantity", "Total_Spent"]:
    lower, upper = iqr_bounds(df[c])
    mask = (df[c] < lower) | (df[c] > upper)
    outlier_summary.append([c, lower, upper, int(mask.sum())])
    df[c + "_Outlier_Flag"] = mask
    df[c] = df[c].clip(lower, upper)

outlier_summary = pd.DataFrame(
    outlier_summary, columns=["column","lower_bound","upper_bound","outlier_count"]
)
display(outlier_summary)


## 4. Feature engineering

In [ ]:
df["Year"] = df["Transaction_Date"].dt.year.astype("Int64")
df["Month"] = df["Transaction_Date"].dt.month.astype("Int64")
df["Month_Name"] = df["Transaction_Date"].dt.month_name()

# Revenue based on standardized price and quantity
df["Revenue"] = (df["Price_Per_Unit"] * df["Quantity"]).round(2)

# No actual cost column exists in the source data.
# Illustrative assumption only: cost = 70% of revenue.
COST_RATE_ASSUMPTION = 0.70
df["Estimated_Cost"] = (df["Revenue"] * COST_RATE_ASSUMPTION).round(2)
df["Estimated_Profit"] = (df["Revenue"] - df["Estimated_Cost"]).round(2)
df["Estimated_Profit_Margin"] = np.where(
    df["Revenue"] != 0,
    (df["Estimated_Profit"] / df["Revenue"]).round(4),
    0
)

display(df.head())


## 5. After-cleaning profile

In [ ]:
profile_after = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=True)
})
display(profile_after)

print("Final shape:", df.shape)
print("Exact duplicate rows after cleaning:", df.duplicated().sum())


## 6. Before vs After summary

In [ ]:
before_missing = int(profile_before["missing_count"].sum())
after_missing = int(profile_after["missing_count"].sum())

comparison = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Total missing cells", "Exact duplicate rows"],
    "Before": [len(pd.read_csv(DATA_FILE)), len(pd.read_csv(DATA_FILE).columns),
               before_missing, int(pd.read_csv(DATA_FILE).duplicated().sum())],
    "After": [len(df), len(df.columns), after_missing, int(df.duplicated().sum())]
})
display(comparison)


## 7. Export cleaned dataset

In [ ]:
OUTPUT_FILE = "clean_dataset.csv"
df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(df):,} rows and {len(df.columns):,} columns to {OUTPUT_FILE}")
display(pd.read_csv(OUTPUT_FILE).head())


## Conclusion

The raw retail dataset contained more than 10,000 rows and intentional missing/inconsistent values. The workflow:

- loaded and profiled the raw data,
- corrected data types,
- removed exact duplicates,
- handled missing categorical and numeric values,
- detected and capped IQR-based numeric outliers,
- extracted year/month features,
- calculated revenue,
- added an explicitly labelled estimated profit/margin proxy,
- and exported the standardized result as `clean_dataset.csv`.

**Assumption disclosure:** Because the source data has no actual cost field, the estimated profit and margin are illustrative only and should be replaced by real cost data for financial reporting.
